In [ ]:
# # Data Exploration and Analysis

# ## Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# ## Load Data

# Load the raw data
df = pd.read_excel("../../data/raw/PUF2023.xlsx")
print(f"Initial Shape: {df.shape}")
df.head()

In [ ]:
# ## Data Overview

# Data types
df.dtypes

# Basic statistics
df.describe()

# Check for missing values
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

In [ ]:
# ## Data Cleaning

# Drop duplicate/redundant columns (j-prefixed)
df = df.drop(columns=df.filter(regex='^j').columns)
print(f"Shape after dropping j columns: {df.shape}")

# Map categorical variables
location_map = {1: 'Urban', 2: 'Suburban', 3: 'Rural'}
region_map = {1: 'Northeast', 2: 'Midwest', 3: 'South', 4: 'West'}
titled_map = {1: 'Vehicle', 2: 'Land-Home', 3: 'Other'}

df['LOCATION'] = df['LOCATION'].map(location_map)
df['REGION'] = df['REGION'].map(region_map)
df['TITLED'] = df['TITLED'].map(titled_map)
df['LEASE'] = df['LEASE'].replace({2: 0, 1: 1})

# Handle missing values
num_cols = ['PRICE', 'SQFT', 'BEDROOMS']
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Filter outliers
df = df[(df['PRICE'] > 10000) & (df['PRICE'] < 1e6)]
df = df[(df['SQFT'] > 100) & (df['SQFT'] < 5000)]
print(f"Shape after cleaning: {df.shape}")

In [ ]:
# ## Exploratory Data Analysis

# ### Target Distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['PRICE'], kde=True, bins=50)
plt.title('Distribution of House Prices')
plt.xlabel('Price')

plt.subplot(1, 2, 2)
sns.boxplot(x=df['PRICE'])
plt.title('Box Plot of House Prices')

plt.tight_layout()
plt.show()

# ### Price vs Square Footage
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(x='SQFT', y='PRICE', data=df, alpha=0.5)
plt.title('Price vs Square Footage')
plt.xlabel('Square Footage')
plt.ylabel('Price')

plt.subplot(1, 2, 2)
sns.scatterplot(x='SQFT', y='PRICE', data=df, hue='LOCATION', alpha=0.5)
plt.title('Price vs Square Footage by Location')
plt.xlabel('Square Footage')
plt.ylabel('Price')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

# ### Price by Categorical Variables
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Price by Location
sns.boxplot(x='LOCATION', y='PRICE', data=df, ax=axes[0, 0])
axes[0, 0].set_title('Price by Location')

# Price by Region
sns.boxplot(x='REGION', y='PRICE', data=df, ax=axes[0, 1])
axes[0, 1].set_title('Price by Region')

# Price by Title
sns.boxplot(x='TITLED', y='PRICE', data=df, ax=axes[1, 0])
axes[1, 0].set_title('Price by Title')

# Price by Lease
sns.boxplot(x='LEASE', y='PRICE', data=df, ax=axes[1, 1])
axes[1, 1].set_title('Price by Lease')

plt.tight_layout()
plt.show()

# ### Correlation Analysis
# Select features for correlation
useful_cols = ['PRICE', 'SQFT', 'BEDROOMS', 'FOOTINGS', 'LEASE']
corr_df = df[useful_cols].copy()
corr_df['PRICE_PER_SQFT'] = corr_df['PRICE'] / corr_df['SQFT']

plt.figure(figsize=(10, 8))
correlation_matrix = corr_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()

# ### Feature Engineering
# Create additional features
df_engineered = df.copy()
df_engineered['PRICE_PER_SQFT'] = df_engineered['PRICE'] / df_engineered['SQFT']
df_engineered['BEDROOMS_PER_SQFT'] = df_engineered['BEDROOMS'] / df_engineered['SQFT'] * 1000
df_engineered['LOG_PRICE'] = np.log1p(df_engineered['PRICE'])
df_engineered['LOG_SQFT'] = np.log1p(df_engineered['SQFT'])

In [ ]:
# ## Save Processed Data
df_engineered.to_csv("../../data/processed/house_prices_processed.csv", index=False)
print("Processed data saved!")